# Codesign: single-network morphology generator (full ant)

Report for `experiments/ant_codesign.py`. Control (ContAct + ContCrit/V0.98) and the morphology
generator (GenAct + GenCrit/V1.0) share **one transformer trunk** under one optimizer. The generator
builds a body one token at a time (random slot order, each slot conditioned on the committed ones) and
is rewarded by body quality `R` = the **true mean episode return** (γ=1) over the window — not the raw
per-step reward. Per step: plain combined PPO on control. At each resample: fit GenCrit/V1.0 (rollout
states + designed prefixes → R), GenAct PPO, and **clone control** (β·KL + λ·MSE) so the shared-trunk
update doesn't drift control. **Gate:** per-leg presence climbs base `[1,4,6]` -> ~all-8, and the
**per-token marginal value** is positive for useful legs (no leg/energy cost yet, so more legs => more
return).

X axis is **epochs**. The dashed line marks the **pretrain->RL handoff** (generator switches from
imitating base±flip to optimizing `R`). Generator metrics (`gen/*`, `gen_marg/*`) are logged once per
resample window, so they begin sparse.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

D = np.load(Path('../data/ant_codesign/curves.npz'))
seeds = list(D['seeds'])
limbs = [str(l) for l in D['limbs']]
BASE = {'F', 'BR', 'BL'}            # base morph [1,4,6]

def series(seed, tag):
    k = f"s{seed}__{tag.replace('/', '_')}"
    return D[k + '__step'], D[k + '__val']

# X axis in EPOCHS. TB logs a global frame counter (num_actors*horizon per epoch); rewards/step is
# logged every epoch, so consecutive step deltas give frames/epoch directly (no config needed).
_mr_step, _ = series(seeds[0], 'rewards/step')
FRAMES_PER_EPOCH = int(np.median(np.diff(_mr_step)))
to_epoch = lambda step: np.asarray(step) / FRAMES_PER_EPOCH

# Pretrain->RL handoff: first window where the generator fraction reaches 1.0 (return-driven on).
_gf_step, _gf_val = series(seeds[0], 'gen/fraction')
RL_ONSET = float(to_epoch(_gf_step[np.argmax(_gf_val >= 1.0)]))
# Resample-window length in epochs (generator logs once per window: R_mean is logged every window).
_gv_step, _ = series(seeds[0], 'quality/R_mean')
WINDOW_EPOCHS = max(1, int(round(np.median(np.diff(to_epoch(_gv_step))))))
print(f'seeds: {seeds} | frames/epoch: {FRAMES_PER_EPOCH} | pretrain->RL @ epoch {RL_ONSET:.0f} '
      f'| window ~{WINDOW_EPOCHS} epochs')

def smooth(y, w):
    """Centered rolling mean, edge-normalized (no end droop)."""
    y = np.asarray(y, float)
    if w is None or w <= 1 or len(y) < 2:
        return y
    k = np.ones(min(w, len(y)))
    return np.convolve(y, k, 'same') / np.convolve(np.ones_like(y), k, 'same')

def band(ax, tag, smooth_w=None, **kw):
    """Mean (+min/max band over seeds) of a tag vs EPOCH, optionally rolling-smoothed."""
    xs, ys = [], []
    for s in seeds:
        try:
            x, y = series(s, tag)
        except KeyError:
            continue
        xs.append(x); ys.append(y)
    if not ys:
        return
    m = min(len(a) for a in ys)
    x = to_epoch(xs[0][:m]); Y = np.stack([a[:m] for a in ys])
    ax.plot(x, smooth(Y.mean(0), smooth_w), **kw)
    if len(ys) > 1:
        ax.fill_between(x, smooth(Y.min(0), smooth_w), smooth(Y.max(0), smooth_w),
                        alpha=0.15, color=kw.get('color'))

def mark_rl(ax):
    """Dashed line + label at the pretrain->RL handoff (the presence inflection)."""
    ax.axvline(RL_ONSET, color='k', lw=1.0, ls='--', alpha=0.7)
    ax.text(RL_ONSET, 0.99, ' pretrain→RL', transform=ax.get_xaxis_transform(),
            va='top', ha='left', fontsize=8)

## 1. Per-leg presence probability — the gate

Base legs (F, BR, BL) bold; all eight should rise toward 1.0 after the pretrain->RL handoff.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
cmap = plt.cm.tab10
for i, limb in enumerate(limbs):
    base = limb in BASE
    band(ax, f'build/p/{limb}', color=cmap(i), lw=2.6 if base else 1.4,
         ls='-' if base else '--', label=f"{limb}{' (base)' if base else ''}")
ax.axhline(1.0, color='k', lw=0.8, ls=':')
ax.set(xlabel='epoch', ylabel='presence prob  p(limb)', ylim=(0, 1.02),
       title='Per-limb presence probability (gate: all -> ~1)')
mark_rl(ax)
ax.legend(ncol=4, fontsize=9); fig.tight_layout()

## 2. Mean leg count

`built/mean_legcount` = the bodies actually built each window (includes the pretrain base±flip ramp;
in the RL phase this equals the generator's per-slot rate summed). Should climb base(~3) -> ~8.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
band(ax, 'build/limbcount', color='C1', lw=2.4, label='built #limbs')
ax.axhline(8, color='k', lw=0.8, ls=':'); ax.axhline(3, color='gray', lw=0.8, ls=':')
ax.set(xlabel='epoch', ylabel='mean limb count', title='Mean limb count (3 base -> 8 optimum)')
mark_rl(ax)
ax.legend(); fig.tight_layout()

## 3. Per-token marginal value per leg — the headline

`gen_marg/{leg}` = the generator's own estimate of the value added by turning leg *k* on, in
`R = V1.0(s0)` units (the raw telescoping advantage `v(prefix+leg) - v(prefix)` averaged over the
window's `on` decisions). **This is the per-token marginal-value credit that escapes ADR-0010's
body-agnostic-baseline trap.** RL-phase only (no advantages during pretrain), so it starts at the
handoff; a leg's curve drops out once that leg is always-on (no `on` decision left to measure).
**Gate: positive for every useful leg.**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
cmap = plt.cm.tab10
for i, limb in enumerate(limbs):
    base = limb in BASE
    band(ax, f'gen/marg/{limb}', color=cmap(i), lw=2.6 if base else 1.4,
         ls='-' if base else '--', label=f"{limb}{' (base)' if base else ''}")
ax.axhline(0, color='k', lw=0.8, ls=':')
ax.set(xlabel='epoch', ylabel='marginal value  v(prefix+limb) − v(prefix)   [R units]',
       title='Per-token marginal value per limb (gate: positive for useful limbs)')
mark_rl(ax)
ax.legend(ncol=4, fontsize=9); fig.tight_layout()

## 4. Body quality `R` and the generator's value-head fit

Two views of the same thing — does the generator's GenCrit/V1.0 head learn the body quality `R` it's
trained toward?

**Left (A), over training, two axes (axis color = line color):**
- **body quality (R)** — blue, left axis — `gen/R_mean`, the mean true episode return (γ=1) per window
  the generator is rewarded by. It drifts *upward* mostly because the control policy keeps improving
  (R is non-stationary), not only because bodies get better.
- **value correlation** — red, right axis — `gen/value_R_corr`, the correlation (0–1) between the
  generator's terminal value `v(full)` and `R` across envs. Closer to 1 = the value head tracks `R`.

**Right (B), final window, per-env scatter:** each point is one env — **x = target** `R`,
**y = prediction** `v(full)` — with `y = x` = perfect calibration and the overall correlation in the
title.

> **Note:** `v(full)` is a function of the **body only** (the committed tokens), not of the control
> state. So if the generator has collapsed to a single body (all envs identical), every point shares
> one `y` and the scatter degenerates to a **horizontal line** (corr → n/a). `R` still spreads on `x`
> from control reset-noise. A flat line here is a **diversity** symptom, not a plotting bug.

In [ ]:
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5))

# --- A: two DIFFERENT quantities over training, one per y-axis (axis color = line color) ---
C_R, C_CORR = 'C0', 'C3'
# left axis (blue): mean body quality R the generator is rewarded by
band(axA, 'quality/R_mean', color=C_R, lw=2.4, label='body quality (R)')
axA.set_xlabel('epoch')
axA.set_ylabel('body quality   R = mean episode return   (reward units)', color=C_R)
axA.tick_params(axis='y', labelcolor=C_R)
# right axis (red): how well the generator's value head predicts R (0..1)
axA2 = axA.twinx()
band(axA2, 'gencrit/value_rank_corr', color=C_CORR, lw=2.0, label='value rank corr (denoised)')
axA2.set_ylabel('value rank corr   Spearman(v, meanR) over bodies   [0–1]', color=C_CORR)
axA2.tick_params(axis='y', labelcolor=C_CORR)
axA2.set_ylim(-0.05, 1.02)
axA2.axhline(1.0, color=C_CORR, lw=0.8, ls=':', alpha=0.5)
mark_rl(axA)
# combined legend across both twin axes
h1, l1 = axA.get_legend_handles_labels(); h2, l2 = axA2.get_legend_handles_labels()
axA.legend(h1 + h2, l1 + l2, loc='center right')
axA.set_title('Over training: body quality (R) and denoised value-head fit to R')

# --- B: final-window per-env calibration scatter (target on x, prediction on y) ---
S = np.load(Path('../data/ant_codesign/gen_scatter.npz'))
Rs, Vs = [], []
for i, s in enumerate(seeds):
    R, vf = S[f's{s}__R'], S[f's{s}__v_full']
    Rs.append(R); Vs.append(vf)
    axB.scatter(R, vf, s=5, alpha=0.25, color=f'C{i}', label=f'seed {s}' if len(seeds) > 1 else 'envs')
R_all, V_all = np.concatenate(Rs), np.concatenate(Vs)
lo = min(R_all.min(), V_all.min()); hi = max(R_all.max(), V_all.max())
axB.plot([lo, hi], [lo, hi], 'k--', lw=1.2, label='perfect calibration (y = x)')
# corr only meaningful with real v(full) spread; one-body collapse => skip (avoids a nan/divide warning)
if R_all.std() > 1e-6 and V_all.std() > 1e-3:
    rtxt = f'{np.corrcoef(R_all, V_all)[0, 1]:.2f}'
else:
    rtxt = 'n/a (all envs built one body → no v spread)'
axB.set_xlabel('target:  body quality R = mean episode return   (what the generator is trained toward)')
axB.set_ylabel('prediction:  generator value v(full)')
axB.set_title(f'Final window: is the generator value calibrated?   (corr = {rtxt})')
axB.legend(loc='upper left', fontsize=9)
fig.tight_layout()

## 5. Control preservation (clone) + GenCrit/V1.0 fit

The resample update touches the **shared trunk**, so control must be held in place while only the
generator side learns. Two clone terms enforce this, logged per window:
- `gen/clone_kl` — KL[ContAct_old ‖ ContAct] on rollout states (β-weighted). Near 0 = the control
  actor barely moved during the generator update.
- `gen/clone_crit_mse` — MSE(ContCrit, ContCrit_old) (λ-weighted). Near 0 = V0.98 held.

And the GenCrit/V1.0 regression loss, fit on two sources toward the same per-body `R`:
- `gen/vloss_prefix` — fit on the designed-token prefixes (generation MDP).
- `gen/vloss_rollout` — fit on live rollout states.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))

# --- L: control-preservation clone terms (lower = control held better) ---
band(axL, 'clone/actor_kl', color='C0', lw=2.2, label='clone KL[ContAct]')
axL.set_xlabel('epoch'); axL.set_ylabel('actor KL', color='C0')
axL.tick_params(axis='y', labelcolor='C0')
axL2 = axL.twinx()
band(axL2, 'clone/critic_mse', color='C3', lw=2.0, label='clone MSE(ContCrit)')
axL2.set_ylabel('critic MSE', color='C3'); axL2.tick_params(axis='y', labelcolor='C3')
h1, l1 = axL.get_legend_handles_labels(); h2, l2 = axL2.get_legend_handles_labels()
axL.legend(h1 + h2, l1 + l2, loc='upper right')
mark_rl(axL); axL.set_title('Control preservation during the generator update (clone)')

# --- R: GenCrit/V1.0 regression loss, both fit sources ---
band(axR, 'gencrit/loss_prefix', color='C2', lw=2.2, label='GenCrit fit: designed prefixes')
band(axR, 'gencrit/loss_rollout', color='C4', lw=2.2, label='GenCrit fit: rollout states')
axR.set(xlabel='epoch', ylabel='MSE / Var(R)  (frac. unexplained)'); axR.legend(loc='upper right')
mark_rl(axR); axR.set_title('GenCrit / V1.0 fit to body quality R (scale-free)')
fig.tight_layout()

## 6. Control performance (training, all bodies)

`rewards/step` = mean episode reward over all envs each epoch (the col-0 / V0.98 return), i.e. control
skill across the mix of bodies actually being trained on. Raw curve is faint; the bold line is a
one-resample-window rolling mean, which averages out the per-rebuild dips. (Complements the
`R = V1.0(s0)` body-quality curve in §4.)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
band(ax, 'rewards/step', color='C2', lw=0.8, alpha=0.3, label='raw (per epoch)')
band(ax, 'rewards/step', color='C2', lw=2.4, smooth_w=WINDOW_EPOCHS,
     label=f'{WINDOW_EPOCHS}-epoch rolling mean')
ax.set(xlabel='epoch', ylabel='mean episode reward', title='Control performance over training')
mark_rl(ax)
ax.legend(); fig.tight_layout()

## 7. Generator entropy + gen fraction

`gen/entropy` is the mean per-token `{on,stop}` policy entropy; a gentle decline (not a cliff) means no
premature collapse. `gen/fraction` is the pretrain->RL ramp: the share of envs built from the generator
(vs base±flip) per window, climbing 0 -> 1 across the pretrain windows.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
band(ax, 'gen/entropy', color='C4', lw=2.0, label='gen/entropy')
ax.set(xlabel='epoch', ylabel='gen/entropy'); ax.legend(loc='upper left')
ax2 = ax.twinx()
band(ax2, 'gen/fraction', color='C6', lw=2.0, label='gen/fraction')
ax2.set_ylabel('gen fraction'); ax2.set_ylim(-0.02, 1.02); ax2.legend(loc='lower right')
mark_rl(ax)
ax.set_title('Generator entropy + gen fraction'); fig.tight_layout()

## 8. Control performance on the BASE morphology

Offline per-checkpoint eval (deterministic mu, raw episode return) on the FIXED base body — isolates
control skill on one body, decoupled from the changing generated bodies. **Rising/flat = base skill
retained; falling = the policy forgets base as it specializes on leggier generated bodies.**

In [ ]:
B = np.load(Path('../data/ant_codesign/base_eval.npz'))
base_lbl = '[' + ','.join(str(int(x)) for x in B['base_legs']) + ']'
fig, ax = plt.subplots(figsize=(9, 5))
for i, s in enumerate(seeds):
    ep, mu, sd = B[f's{s}__epoch'], B[f's{s}__ret_mean'], B[f's{s}__ret_std']
    ax.plot(ep, mu, color='C5', lw=2.2, label=f'base {base_lbl} return' if i == 0 else None)
    ax.fill_between(ep, mu - sd, mu + sd, color='C5', alpha=0.15)
ax.set(xlabel='epoch', ylabel='episode return',
       title=f'Control performance on the base morphology {base_lbl} (deterministic)')
mark_rl(ax)
ax.legend(); fig.tight_layout()